In [1]:
!cd .. && pip install .

Processing /home/jovyan/work
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for massi: filename=massi-0.1.0-py3-none-any.whl size=16753 sha256=abd8cf2b4953d97c74fecc5f9034a0a7d38d50f7ee6112f113ede1c845251cf2
  Stored in directory: /tmp/pip-ephem-wheel-cache-4gnm8yi1/wheels/b3/ef/92/093562baef6fce9b92cfc59927bf9e82e306ed148e873389c0
Successfully built massi
  Attempting uninstall: massi
    Found existing installation: massi 0.1.0
    Uninstalling massi-0.1.0:
      Successfully uninstalled massi-0.1.0


1. **We will use the relay model for the sake of debugging convenience**

In [2]:
from massi.agents.massi import MassiScenarioFactory, Config, Dependencies, StandardIOPromptClient
from massi.agents.stunts import RelayChatModel

In [3]:
config = Config(**{"model_url": "http://host.docker.internal:11434", "model_name": "granite4:3b"})
deps = Dependencies(llm_client=RelayChatModel(), prompt_client=StandardIOPromptClient())

In [4]:
factory = MassiScenarioFactory(config)

In [5]:
graph = factory.resource_property_graph(deps)

In [6]:
import grandalf

graph.get_graph().print_ascii()

   +-----------+     
   | __start__ |     
   +-----------+     
          *          
          *          
          *          
+----------------+   
| scan_directory |   
+----------------+   
          *          
          *          
          *          
+-----------------+  
| score_relevancy |  
+-----------------+  
          *          
          *          
          *          
    +---------+      
    | __end__ |      
    +---------+      


In [7]:
import random

thread = {"configurable": {"thread_id": str(random.randint(1, 100))}}

In [8]:
import pprint

In [9]:
def stream_until_iterruption(graph, *args):
    for event in graph.stream(*args, stream_mode="values"):
        print("\n==== new event ====")
        for k, v in event.items():
            print(f"---- {k} ----")
            pprint.pp(v, indent=4)

In [10]:
DEFAULT_SCORING_RESPONSE = """
{
  "resources": [
    {
      "file_location": "uc1-massi_demo--missing_data.ipynb",
      "filename": "uc1-massi_demo--missing_data.ipynb",
      "extension": ".ipynb",
      "author": "jovyan",
      "gis_relevancy_score": 0.8
    },
    {
      "file_location": "base_metadata.json",
      "filename": "base_metadata.json",
      "extension": ".json",
      "author": "jovyan",
      "gis_relevancy_score": 0.5
    },
    {
      "file_location": "uc2-massi_demo--resource_inference.ipynb",
      "filename": "uc2-massi_demo--resource_inference.ipynb",
      "extension": ".ipynb",
      "author": "jovyan",
      "gis_relevancy_score": 0.8
    },
    {
      "file_location": "massi_demo--print_ascii-test.ipynb",
      "filename": "massi_demo--print_ascii-test.ipynb",
      "extension": ".ipynb",
      "author": "jovyan",
      "gis_relevancy_score": 0.8
    },
    {
      "file_location": ".ipynb_checkpoints/uc1-massi_demo--missing_data-checkpoint.ipynb",
      "filename": "uc1-massi_demo--missing_data-checkpoint.ipynb",
      "extension": ".ipynb",
      "author": "jovyan",
      "gis_relevancy_score": 0.1
    },
    {
      "file_location": ".ipynb_checkpoints/base_metadata-checkpoint.json",
      "filename": "base_metadata-checkpoint.json",
      "extension": ".json",
      "author": "jovyan",
      "gis_relevancy_score": 0.1
    },
    {
      "file_location": ".ipynb_checkpoints/uc2-massi_demo--resource_inference-checkpoint.ipynb",
      "filename": "uc2-massi_demo--resource_inference-checkpoint.ipynb",
      "extension": ".ipynb",
      "author": "jovyan",
      "gis_relevancy_score": 0.1
    },
    {
      "file_location": ".ipynb_checkpoints/massi_demo--print_ascii-test-checkpoint.ipynb",
      "filename": "massi_demo--print_ascii-test-checkpoint.ipynb",
      "extension": ".ipynb",
      "author": "jovyan",
      "gis_relevancy_score": 0.1
    }
  ]
}
"""

In [11]:
from unittest.mock import patch

# Run the graph until the first interruption
with patch('builtins.input', return_value=DEFAULT_SCORING_RESPONSE):
    stream_until_iterruption(graph, {}, thread)


==== new event ====
---- messages ----
[]
---- files ----
[   ResourceFile(file_location='uc1-massi_demo--missing_data.ipynb', filename='uc1-massi_demo--missing_data.ipynb', extension='.ipynb', author='jovyan'),
    ResourceFile(file_location='base_metadata.json', filename='base_metadata.json', extension='.json', author='jovyan'),
    ResourceFile(file_location='uc2-massi_demo--resource_inference.ipynb', filename='uc2-massi_demo--resource_inference.ipynb', extension='.ipynb', author='jovyan'),
    ResourceFile(file_location='massi_demo--print_ascii-test.ipynb', filename='massi_demo--print_ascii-test.ipynb', extension='.ipynb', author='jovyan'),
    ResourceFile(file_location='.ipynb_checkpoints/uc1-massi_demo--missing_data-checkpoint.ipynb', filename='uc1-massi_demo--missing_data-checkpoint.ipynb', extension='.ipynb', author='jovyan'),
    ResourceFile(file_location='.ipynb_checkpoints/base_metadata-checkpoint.json', filename='base_metadata-checkpoint.json', extension='.json', author=

In [12]:
type(graph.get_state(thread).values["gis_scored_files"])

massi.agents.constructs.GISScoringCollection

In [13]:
graph.get_state(thread).values["gis_scored_files"]

GISScoringCollection(resources=[GISScoredFile(file_location='uc1-massi_demo--missing_data.ipynb', filename='uc1-massi_demo--missing_data.ipynb', extension='.ipynb', author='jovyan', gis_relevancy_score=0.8), GISScoredFile(file_location='base_metadata.json', filename='base_metadata.json', extension='.json', author='jovyan', gis_relevancy_score=0.5), GISScoredFile(file_location='uc2-massi_demo--resource_inference.ipynb', filename='uc2-massi_demo--resource_inference.ipynb', extension='.ipynb', author='jovyan', gis_relevancy_score=0.8), GISScoredFile(file_location='massi_demo--print_ascii-test.ipynb', filename='massi_demo--print_ascii-test.ipynb', extension='.ipynb', author='jovyan', gis_relevancy_score=0.8), GISScoredFile(file_location='.ipynb_checkpoints/uc1-massi_demo--missing_data-checkpoint.ipynb', filename='uc1-massi_demo--missing_data-checkpoint.ipynb', extension='.ipynb', author='jovyan', gis_relevancy_score=0.1), GISScoredFile(file_location='.ipynb_checkpoints/base_metadata-checkp

In [14]:
graph.get_state(thread).values["gis_scored_files"].resources

[GISScoredFile(file_location='uc1-massi_demo--missing_data.ipynb', filename='uc1-massi_demo--missing_data.ipynb', extension='.ipynb', author='jovyan', gis_relevancy_score=0.8),
 GISScoredFile(file_location='base_metadata.json', filename='base_metadata.json', extension='.json', author='jovyan', gis_relevancy_score=0.5),
 GISScoredFile(file_location='uc2-massi_demo--resource_inference.ipynb', filename='uc2-massi_demo--resource_inference.ipynb', extension='.ipynb', author='jovyan', gis_relevancy_score=0.8),
 GISScoredFile(file_location='massi_demo--print_ascii-test.ipynb', filename='massi_demo--print_ascii-test.ipynb', extension='.ipynb', author='jovyan', gis_relevancy_score=0.8),
 GISScoredFile(file_location='.ipynb_checkpoints/uc1-massi_demo--missing_data-checkpoint.ipynb', filename='uc1-massi_demo--missing_data-checkpoint.ipynb', extension='.ipynb', author='jovyan', gis_relevancy_score=0.1),
 GISScoredFile(file_location='.ipynb_checkpoints/base_metadata-checkpoint.json', filename='base